In [10]:



import os
import sys
import csv
import json
import random
import hashlib
from pathlib import Path
from collections import Counter

# ============================================================
# GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, roc_auc_score, roc_curve, auc
)

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import imagehash
except ImportError:
    import subprocess
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "ImageHash", "-q"]
    )
    import imagehash


# ============================================================
# CONFIGURATION
# ============================================================

# CHANGE THIS PATH according to your Google Drive folder.
# Example:
# /content/drive/MyDrive/Brain MRI Dataset

DATASET_DIR = "/content/drive/MyDrive/dataset/Stroke_classification"

SOURCE_DIR = Path(DATASET_DIR)

OUTPUT_DIR = "/content/drive/MyDrive/ViT_Results"

# Class folder names -> labels
CLASS_FOLDERS = {
    "Normal": 0,
    "Haemorrhagic": 1,
    "Ischemic": 2,
}

CLASS_NAMES = ["Normal", "Haemorrhagic", "Ischemic"]

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
}

IMAGE_SIZE = 224
BATCH_SIZE = 16
MAX_EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

SEEDS = [42, 123, 2026]

PHASH_MAX_DISTANCE = 8
EMBEDDING_COSINE_THRESHOLD = 0.95

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA AUDIT - GOOGLE DRIVE
# ============================================================

def setup_dataset_from_drive():

    global SOURCE_DIR

    SOURCE_DIR = Path(DATASET_DIR)

    if not SOURCE_DIR.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {SOURCE_DIR}\n"
            "Please check your Google Drive folder path."
        )

    print("\nGoogle Drive dataset location:")
    print(SOURCE_DIR)

    print("\nChecking class folders...")

    for class_name in CLASS_FOLDERS:

        class_path = SOURCE_DIR / class_name

        if not class_path.exists():
            raise FileNotFoundError(
                f"Class folder not found: {class_path}\n"
                f"Expected folder name: {class_name}"
            )

        print(f"{class_name}: {class_path}")

    print("\nDataset successfully connected from Google Drive!")


def collect_images():

    if SOURCE_DIR is None:
        raise RuntimeError(
            "Dataset has not been connected. "
            "Call setup_dataset_from_drive() first."
        )

    records = []

    for class_name, label in CLASS_FOLDERS.items():

        class_dir = SOURCE_DIR / class_name

        for path in class_dir.rglob("*"):

            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:

                try:
                    with Image.open(path) as img:

                        width, height = img.size
                        img.verify()

                    records.append({
                        "path": str(path),
                        "label": label,
                        "class_name": class_name,
                        "width": width,
                        "height": height,
                    })

                except Exception as exc:

                    print(
                        f"[WARNING] Skipping unreadable image "
                        f"{path}: {exc}"
                    )

    if not records:
        raise RuntimeError(
            "No valid images found. Check your dataset folders."
        )

    df = pd.DataFrame(records)

    counts = (
        df["class_name"]
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
    )

    print("\nDataset audit")
    print("-------------")
    print(counts.to_string())

    print(f"\nTotal valid images: {len(df)}")

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    df.to_csv(
        Path(OUTPUT_DIR) / "dataset_audit.csv",
        index=False
    )

    return df


# ============================================================
# IMAGE DATASET
# ============================================================

class MRIImageDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, int(row["label"]), row["path"]


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.10, 0.10)
    ),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# ============================================================
# 70/15/15 RANDOM IMAGE-LEVEL SPLIT (PROTOCOL A)
# ============================================================

def random_image_split(df, seed):

    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=seed,
        stratify=df["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=seed,
        stratify=temp_df["label"]
    )

    return (
        train_df.copy(),
        val_df.copy(),
        test_df.copy()
    )


# ============================================================
# pHASH + FROZEN RESNET EMBEDDINGS
# ============================================================

def calculate_phashes(df):

    hashes = []

    for path in df["path"]:

        with Image.open(path) as img:
            hashes.append(
                imagehash.phash(img.convert("RGB"))
            )

    return hashes


def create_embedding_model():

    weights = ResNet18_Weights.DEFAULT

    model = models.resnet18(weights=weights)

    model.fc = nn.Identity()

    model.eval().to(DEVICE)

    return model


def calculate_embeddings(df):

    model = create_embedding_model()

    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        ),
    ])

    dataset = MRIImageDataset(
        df,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    all_features = []

    with torch.no_grad():

        for images, _, _ in loader:

            images = images.to(DEVICE)

            features = model(images)

            features = nn.functional.normalize(
                features,
                p=2,
                dim=1
            )

            all_features.append(
                features.cpu().numpy()
            )

    return np.concatenate(all_features, axis=0)


class UnionFind:

    def __init__(self, n):

        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):

        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )

        return self.parent[x]

    def union(self, a, b):

        ra, rb = self.find(a), self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def build_similarity_groups(df):

    print("\nCalculating perceptual hashes...")

    phashes = calculate_phashes(df)

    print("Calculating frozen ResNet-18 embeddings...")

    embeddings = calculate_embeddings(df)

    n = len(df)

    uf = UnionFind(n)

    phash_values = [
        int(str(h), 16)
        for h in phashes
    ]

    print(
        "Linking exact/near duplicates "
        "and similar embeddings..."
    )

    for i in range(n):

        for j in range(i + 1, n):

            hamming_distance = (
                phash_values[i] ^ phash_values[j]
            ).bit_count()

            cosine_similarity = float(
                np.dot(embeddings[i], embeddings[j])
            )

            if (
                hamming_distance <= PHASH_MAX_DISTANCE
                or cosine_similarity >= EMBEDDING_COSINE_THRESHOLD
            ):

                uf.union(i, j)

    roots = [uf.find(i) for i in range(n)]

    root_to_group = {}
    group_ids = []

    for root in roots:

        if root not in root_to_group:
            root_to_group[root] = len(root_to_group)

        group_ids.append(root_to_group[root])

    result = df.copy()

    result["group_id"] = group_ids

    group_sizes = result.groupby("group_id").size()

    print(f"Images: {n}")
    print(f"Approximate groups: {len(group_sizes)}")
    print(f"Median group size: {group_sizes.median():.1f}")
    print(f"Largest group: {group_sizes.max()}")

    result.to_csv(
        Path(OUTPUT_DIR) / "image_groups.csv",
        index=False
    )

    return result


# ============================================================
# APPROXIMATE STRATIFIED GROUP 70/15/15 SPLIT
# PROTOCOL B
# ============================================================

def stratified_group_split(df, seed):

    rng = np.random.default_rng(seed)

    group_table = (
        df.groupby("group_id")["label"]
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=[0, 1, 2], fill_value=0)
    )

    group_ids = group_table.index.to_list()

    vectors = group_table.to_numpy(dtype=float)

    total_by_class = vectors.sum(axis=0)

    targets = np.array(
        [0.70, 0.15, 0.15]
    )[:, None] * total_by_class[None, :]

    tie_break = rng.random(len(group_ids))

    order = sorted(
        range(len(group_ids)),
        key=lambda i: (
            -vectors[i].sum(),
            tie_break[i]
        )
    )

    assigned = [[], [], []]

    current = np.zeros((3, 3), dtype=float)

    proportions = np.array([0.70, 0.15, 0.15])

    for idx in order:

        group_vector = vectors[idx]

        best_split = None
        best_score = None

        for split_idx in range(3):

            candidate = current.copy()

            candidate[split_idx] += group_vector

            class_error = np.sum(
                (
                    (candidate - targets)
                    / np.maximum(total_by_class, 1)
                ) ** 2
            )

            total_target = proportions * len(df)

            total_now = candidate.sum(axis=1)

            size_error = np.sum(
                (
                    (total_now - total_target)
                    / max(len(df), 1)
                ) ** 2
            )

            score = class_error + size_error

            if best_score is None or score < best_score:

                best_score = score
                best_split = split_idx

        assigned[best_split].append(group_ids[idx])

        current[best_split] += group_vector

    split_frames = []

    for group_list in assigned:

        split_frames.append(
            df[df["group_id"].isin(group_list)].copy()
        )

    train_df, val_df, test_df = split_frames

    for split_name, part in zip(
        ["train", "validation", "test"],
        [train_df, val_df, test_df]
    ):

        present = set(part["label"].unique())

        if present != {0, 1, 2}:

            print(
                f"[WARNING] {split_name} split does not "
                f"contain all classes. Present labels: "
                f"{sorted(present)}."
            )

    return train_df, val_df, test_df


def save_splits(
    protocol,
    seed,
    train_df,
    val_df,
    test_df
):

    folder = (
        Path(OUTPUT_DIR)
        / protocol
        / f"seed_{seed}"
    )

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    for name, frame in [
        ("train.csv", train_df),
        ("validation.csv", val_df),
        ("test.csv", test_df),
    ]:

        frame.to_csv(
            folder / name,
            index=False
        )

    return folder


# ============================================================
# MODEL TRAINING AND EVALUATION
# ============================================================

def create_resnet18(num_classes=3):

    weights = ResNet18_Weights.DEFAULT

    model = models.resnet18(weights=weights)

    model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )

    return model.to(DEVICE)


def make_loaders(train_df, val_df, test_df):

    train_ds = MRIImageDataset(
        train_df,
        train_transform
    )

    val_ds = MRIImageDataset(
        val_df,
        eval_transform
    )

    test_ds = MRIImageDataset(
        test_df,
        eval_transform
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    return train_loader, val_loader, test_loader


def evaluate_loss(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():

        for images, labels, _ in loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)

            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)

            total_count += labels.size(0)

    return total_loss / max(total_count, 1)


def train_resnet(
    train_df,
    val_df,
    test_df,
    seed,
    save_folder,
    weighted=False
):

    set_seed(seed)

    run_suffix = "_weighted" if weighted else ""

    train_loader, val_loader, test_loader = make_loaders(
        train_df,
        val_df,
        test_df
    )

    model = create_resnet18(3)

    criterion = nn.CrossEntropyLoss()

    if weighted:

        counts = np.bincount(
            train_df["label"].to_numpy(),
            minlength=3
        )

        weights = (
            len(train_df)
            / (3.0 * np.maximum(counts, 1))
        )

        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(
                weights,
                dtype=torch.float32,
                device=DEVICE
            )
        )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(MAX_EPOCHS):

        model.train()

        running_loss = 0.0
        seen = 0

        for images, labels, _ in train_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(logits, labels)

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item() * labels.size(0)
            )

            seen += labels.size(0)

        train_loss = running_loss / max(seen, 1)

        val_loss = evaluate_loss(
            model,
            val_loader,
            nn.CrossEntropyLoss()
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": val_loss,
        })

        print(
            f"Seed {seed} | Epoch {epoch + 1}/{MAX_EPOCHS} "
            f"| train loss {train_loss:.4f} "
            f"| val loss {val_loss:.4f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

            if patience_counter >= PATIENCE:

                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save(
        model.state_dict(),
        save_folder
        / f"resnet18_seed_{seed}{run_suffix}.pth"
    )

    pd.DataFrame(history).to_csv(
        save_folder
        / f"training_history_seed_{seed}{run_suffix}.csv",
        index=False
    )

    y_true, y_pred, y_prob, paths = predict_model(
        model,
        test_loader
    )

    metrics = calculate_metrics(
        y_true,
        y_pred,
        y_prob
    )

    save_evaluation(
        y_true,
        y_pred,
        y_prob,
        paths,
        metrics,
        save_folder,
        f"resnet18_seed_{seed}{run_suffix}"
    )

    return metrics


def predict_model(model, loader):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []
    paths = []

    with torch.no_grad():

        for images, labels, image_paths in loader:

            images = images.to(DEVICE)

            logits = model(images)

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            y_true.extend(
                labels.numpy().tolist()
            )

            y_pred.extend(
                predictions.cpu().numpy().tolist()
            )

            y_prob.extend(
                probabilities.cpu().numpy().tolist()
            )

            paths.extend(list(image_paths))

    return (
        np.array(y_true),
        np.array(y_pred),
        np.array(y_prob),
        paths
    )


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(y_true, y_pred, y_prob=None):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    (
        per_class_precision,
        per_class_recall,
        per_class_f1,
        _
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        zero_division=0
    )

    specificity = []

    for class_idx in range(3):

        tp = cm[class_idx, class_idx]

        fn = cm[class_idx, :].sum() - tp

        fp = cm[:, class_idx].sum() - tp

        tn = cm.sum() - tp - fn - fp

        specificity.append(
            tn / (tn + fp) if (tn + fp) else 0.0
        )

    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": float(
            np.mean(per_class_precision)
        ),
        "macro_recall": float(
            np.mean(per_class_recall)
        ),
        "macro_specificity": float(
            np.mean(specificity)
        ),
        "macro_f1": float(
            np.mean(per_class_f1)
        ),
    }

    for i, name in enumerate(CLASS_NAMES):

        result[f"{name}_precision"] = float(
            per_class_precision[i]
        )

        result[f"{name}_recall"] = float(
            per_class_recall[i]
        )

        result[f"{name}_specificity"] = float(
            specificity[i]
        )

        result[f"{name}_f1"] = float(
            per_class_f1[i]
        )

    if y_prob is not None and len(np.unique(y_true)) == 3:

        y_true_binary = label_binarize(
            y_true,
            classes=[0, 1, 2]
        )

        try:

            result["macro_auc_ovr"] = roc_auc_score(
                y_true_binary,
                y_prob,
                average="macro",
                multi_class="ovr"
            )

            aucs = roc_auc_score(
                y_true_binary,
                y_prob,
                average=None
            )

            for i, name in enumerate(CLASS_NAMES):

                result[f"{name}_auc_ovr"] = float(
                    aucs[i]
                )

        except ValueError:

            result["macro_auc_ovr"] = np.nan

    else:

        result["macro_auc_ovr"] = np.nan

    return result


def save_evaluation(
    y_true,
    y_pred,
    y_prob,
    paths,
    metrics,
    folder,
    prefix
):

    with open(
        folder / f"{prefix}_metrics.json",
        "w"
    ) as f:

        json.dump(metrics, f, indent=4)

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )

    with open(
        folder / f"{prefix}_classification_report.txt",
        "w"
    ) as f:

        f.write(report)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(
        f"Confusion Matrix - {prefix}"
    )

    plt.tight_layout()

    plt.savefig(
        folder / f"{prefix}_confusion_matrix.png",
        dpi=200
    )

    plt.close()

    error_rows = []

    for true_label, pred_label, path in zip(
        y_true,
        y_pred,
        paths
    ):

        if true_label != pred_label:

            error_rows.append({
                "image_path": path,
                "true_class": CLASS_NAMES[int(true_label)],
                "predicted_class": CLASS_NAMES[int(pred_label)],
            })

    pd.DataFrame(error_rows).to_csv(
        folder / f"{prefix}_misclassified.csv",
        index=False
    )

    if y_prob is not None and len(np.unique(y_true)) == 3:

        y_binary = label_binarize(
            y_true,
            classes=[0, 1, 2]
        )

        plt.figure(figsize=(6, 5))

        plotted = False

        for class_idx, class_name in enumerate(CLASS_NAMES):

            if len(np.unique(y_binary[:, class_idx])) < 2:
                continue

            fpr, tpr, _ = roc_curve(
                y_binary[:, class_idx],
                y_prob[:, class_idx]
            )

            class_auc = auc(fpr, tpr)

            plt.plot(
                fpr,
                tpr,
                label=f"{class_name} (AUC={class_auc:.3f})"
            )

            plotted = True

        if plotted:

            plt.plot(
                [0, 1],
                [0, 1],
                linestyle="--"
            )

            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")

            plt.title(
                f"One-vs-rest ROC - {prefix}"
            )

            plt.legend()
            plt.tight_layout()

            plt.savefig(
                folder / f"{prefix}_roc_curves.png",
                dpi=200
            )

        plt.close()


# ============================================================
# BASELINES
# ============================================================

def extract_frozen_features(dataframe):

    model = create_embedding_model()

    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        ),
    ])

    dataset = MRIImageDataset(
        dataframe,
        transform
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    features = []
    labels = []
    paths = []

    with torch.no_grad():

        for images, batch_labels, batch_paths in loader:

            output = model(images.to(DEVICE))

            features.append(
                output.cpu().numpy()
            )

            labels.extend(
                batch_labels.numpy().tolist()
            )

            paths.extend(
                list(batch_paths)
            )

    return (
        np.concatenate(features),
        np.array(labels),
        paths
    )


def run_baselines(
    train_df,
    val_df,
    test_df,
    save_folder
):

    majority_label = int(
        train_df["label"].value_counts().idxmax()
    )

    y_true = test_df["label"].to_numpy()

    majority_pred = np.full(
        len(test_df),
        majority_label
    )

    majority_metrics = calculate_metrics(
        y_true,
        majority_pred
    )

    with open(
        save_folder / "majority_baseline_metrics.json",
        "w"
    ) as f:

        json.dump(
            majority_metrics,
            f,
            indent=4
        )

    train_x, train_y, _ = extract_frozen_features(
        train_df
    )

    val_x, val_y, _ = extract_frozen_features(
        val_df
    )

    test_x, test_y, test_paths = extract_frozen_features(
        test_df
    )

    best_model = None
    best_val_f1 = -1
    best_c = None

    for c_value in [0.01, 0.1, 1, 10]:

        classifier = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=c_value,
                max_iter=3000,
                class_weight=None,
                random_state=42
            )
        )

        classifier.fit(
            train_x,
            train_y
        )

        val_pred = classifier.predict(val_x)

        val_f1 = precision_recall_fscore_support(
            val_y,
            val_pred,
            average="macro",
            zero_division=0
        )[2]

        if val_f1 > best_val_f1:

            best_val_f1 = val_f1
            best_model = classifier
            best_c = c_value

    test_pred = best_model.predict(test_x)

    test_prob = best_model.predict_proba(test_x)

    frozen_metrics = calculate_metrics(
        test_y,
        test_pred,
        test_prob
    )

    with open(
        save_folder / "frozen_logistic_metrics.json",
        "w"
    ) as f:

        json.dump(
            frozen_metrics,
            f,
            indent=4
        )

    with open(
        save_folder / "frozen_logistic_selected_C.txt",
        "w"
    ) as f:

        f.write(
            f"Selected C: {best_c}\n"
        )

    save_evaluation(
        test_y,
        test_pred,
        test_prob,
        test_paths,
        frozen_metrics,
        save_folder,
        "frozen_logistic"
    )


# ============================================================
# LEAKAGE OVERLAP CHECK
# ============================================================

def count_test_train_near_duplicates(
    train_df,
    test_df
):

    combined = pd.concat(
        [train_df, test_df],
        ignore_index=True
    )

    hashes = calculate_phashes(combined)

    embeddings = calculate_embeddings(combined)

    n_train = len(train_df)

    overlap_test_indices = set()

    hash_values = [
        int(str(h), 16)
        for h in hashes
    ]

    for test_idx in range(
        n_train,
        len(combined)
    ):

        for train_idx in range(n_train):

            hamming = (
                hash_values[test_idx]
                ^ hash_values[train_idx]
            ).bit_count()

            cosine = float(
                np.dot(
                    embeddings[test_idx],
                    embeddings[train_idx]
                )
            )

            if (
                hamming <= PHASH_MAX_DISTANCE
                or cosine >= EMBEDDING_COSINE_THRESHOLD
            ):

                overlap_test_indices.add(
                    test_idx - n_train
                )

                break

    return len(overlap_test_indices)


# ============================================================
# MAIN EXPERIMENT
# ============================================================

def main():

    Path(OUTPUT_DIR).mkdir(
        parents=True,
        exist_ok=True
    )

    print("Device:", DEVICE)

    # Load dataset directly from Google Drive
    setup_dataset_from_drive()

    df = collect_images()

    grouped_df = build_similarity_groups(df)

    all_summary_rows = []

    for seed in SEEDS:

        print(
            f"\n{'=' * 60}\n"
            f"SEED: {seed}\n"
            f"{'=' * 60}"
        )

        # Protocol A
        a_train, a_val, a_test = random_image_split(
            df,
            seed
        )

        a_folder = save_splits(
            "protocol_A_random_image",
            seed,
            a_train,
            a_val,
            a_test
        )

        # Protocol B
        b_train, b_val, b_test = stratified_group_split(
            grouped_df,
            seed
        )

        b_folder = save_splits(
            "protocol_B_leakage_aware",
            seed,
            b_train,
            b_val,
            b_test
        )

        # Verify group disjointness
        train_groups = set(b_train["group_id"])
        val_groups = set(b_val["group_id"])
        test_groups = set(b_test["group_id"])

        assert train_groups.isdisjoint(val_groups)
        assert train_groups.isdisjoint(test_groups)
        assert val_groups.isdisjoint(test_groups)

        overlap_a = count_test_train_near_duplicates(
            a_train,
            a_test
        )

        overlap_b = count_test_train_near_duplicates(
            b_train,
            b_test
        )

        pd.DataFrame([
            {
                "protocol": "A_random_image",
                "seed": seed,
                "test_images": len(a_test),
                "test_images_with_near_duplicate_in_train": overlap_a,
                "share_percent": (
                    100 * overlap_a / max(len(a_test), 1)
                ),
            },
            {
                "protocol": "B_leakage_aware",
                "seed": seed,
                "test_images": len(b_test),
                "test_images_with_near_duplicate_in_train": overlap_b,
                "share_percent": (
                    100 * overlap_b / max(len(b_test), 1)
                ),
            },
        ]).to_csv(
            Path(OUTPUT_DIR) / f"overlap_seed_{seed}.csv",
            index=False
        )

        # Baselines under Protocol B
        print("\nRunning Protocol B baselines...")

        run_baselines(
            b_train,
            b_val,
            b_test,
            b_folder
        )

        # Fine-tuned ResNet-18
        for protocol_name, tr, va, te, folder in [
            (
                "A_random_image",
                a_train,
                a_val,
                a_test,
                a_folder
            ),
            (
                "B_leakage_aware",
                b_train,
                b_val,
                b_test,
                b_folder
            ),
        ]:

            print(
                f"\nTraining fine-tuned ResNet-18: "
                f"{protocol_name}"
            )

            metrics = train_resnet(
                tr,
                va,
                te,
                seed,
                folder,
                weighted=False
            )

            row = {
                "protocol": protocol_name,
                "seed": seed,
                **metrics,
            }

            all_summary_rows.append(row)

            # Optional weighted experiment
            if protocol_name == "B_leakage_aware":

                class_shares = tr[
                    "label"
                ].value_counts(normalize=True)

                smallest_share = min([
                    class_shares.get(i, 0.0)
                    for i in range(3)
                ])

                if smallest_share < 0.30:

                    print(
                        "\nRunning class-weighted "
                        "Protocol B experiment..."
                    )

                    weighted_metrics = train_resnet(
                        tr,
                        va,
                        te,
                        seed,
                        folder,
                        weighted=True
                    )

                    all_summary_rows.append({
                        "protocol": "B_leakage_aware_weighted",
                        "seed": seed,
                        **weighted_metrics,
                    })

                else:

                    print(
                        f"Smallest class share is "
                        f"{smallest_share:.1%}; "
                        "weighted-loss experiment skipped."
                    )

    summary = pd.DataFrame(all_summary_rows)

    summary.to_csv(
        Path(OUTPUT_DIR) / "all_seed_results.csv",
        index=False
    )

    metric_columns = [
        c for c in summary.columns
        if c not in ["protocol", "seed"]
    ]

    aggregate = summary.groupby(
        "protocol"
    )[metric_columns].agg(["mean", "std"])

    aggregate.to_csv(
        Path(OUTPUT_DIR) / "mean_std_results.csv"
    )

    print("\nAll experiments completed.")

    print("Results saved to:", OUTPUT_DIR)

    print("\nMean ± standard deviation:")

    print(aggregate)

    # Package results
    import shutil

    results_zip = shutil.make_archive(
        OUTPUT_DIR,
        "zip",
        OUTPUT_DIR
    )

    print("\nResults ZIP created:", results_zip)

    try:

        from google.colab import files

        files.download(results_zip)

    except Exception as exc:

        print(
            "Automatic download could not start. "
            "Results remain in:",
            OUTPUT_DIR,
            exc
        )


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu

Google Drive dataset location:
/content/drive/MyDrive/dataset/Stroke_classification

Checking class folders...
Normal: /content/drive/MyDrive/dataset/Stroke_classification/Normal
Haemorrhagic: /content/drive/MyDrive/dataset/Stroke_classification/Haemorrhagic
Ischemic: /content/drive/MyDrive/dataset/Stroke_classification/Ischemic

Dataset successfully connected from Google Drive!

Dataset audit
-------------
class_name
Normal          399
Haemorrhagic    186
Ischemic         30

Total valid images: 615

Calculating perceptual hashes...
Calculating frozen ResNet-18 embeddings...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 59.0MB/s]


Linking exact/near duplicates and similar embeddings...
Images: 615
Approximate groups: 46
Median group size: 1.0
Largest group: 526

SEED: 42

Running Protocol B baselines...

Training fine-tuned ResNet-18: A_random_image
Seed 42 | Epoch 1/30 | train loss 0.4796 | val loss 0.3303
Seed 42 | Epoch 2/30 | train loss 0.1923 | val loss 0.3026
Seed 42 | Epoch 3/30 | train loss 0.1108 | val loss 0.1937
Seed 42 | Epoch 4/30 | train loss 0.1227 | val loss 0.1274
Seed 42 | Epoch 5/30 | train loss 0.0327 | val loss 0.1380
Seed 42 | Epoch 6/30 | train loss 0.0442 | val loss 0.2772
Seed 42 | Epoch 7/30 | train loss 0.0285 | val loss 0.1520
Seed 42 | Epoch 8/30 | train loss 0.0092 | val loss 0.2125
Seed 42 | Epoch 9/30 | train loss 0.0123 | val loss 0.0949
Seed 42 | Epoch 10/30 | train loss 0.0104 | val loss 0.1612
Seed 42 | Epoch 11/30 | train loss 0.0834 | val loss 0.4453
Seed 42 | Epoch 12/30 | train loss 0.0841 | val loss 0.3505
Seed 42 | Epoch 13/30 | train loss 0.0756 | val loss 0.5766
Seed 4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>